# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent)

**Quy ước ma trận:**
- `0` → Ô trống (sạch) / Máy hút bụi
- `1` → Bụi

**Mở rộng:** Bổ sung thuật toán Iterative Deepening Search (IDS) chạy theo 2 cách:
1. **Early Goal Test:** Xét điều kiện rồi mới đưa vào frontier.
2. **Late Goal Test:** Đưa vào frontier, khi nào pop ra mới xét tiếp các nút con.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap


In [ ]:
# ── Cấu hình ──
ROWS      = 5
COLS      = 7
DUST_PROB = 0.4

# ── Tạo môi trường ──
def create_env(rows, cols, dust_prob):
    grid = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            if random.random() < dust_prob:
                grid[r][c] = 1
    return grid

# ── Vẽ ma trận ──
def draw_grid(grid, pos, title):
    rows, cols = grid.shape
    fig, ax = plt.subplots(figsize=(cols * 0.9, rows * 0.9))

    cmap = ListedColormap(['#F0F0F0', '#F4D03F'])  # 0=xám nhạt, 1=vàng
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=1)

    for r in range(rows):
        for c in range(cols):
            if (r, c) == pos:
                ax.text(c, r, '🤖', ha='center', va='center', fontsize=14)
            elif grid[r][c] == 1:
                ax.text(c, r, '●', ha='center', va='center',
                        fontsize=16, color='#884400')
            else:
                ax.text(c, r, '0', ha='center', va='center',
                        fontsize=11, color='#888888')

    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    ax.set_xticklabels(np.arange(cols))
    ax.set_yticklabels(np.arange(rows))
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which='minor', color='#AAAAAA', linewidth=0.8)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)

    legend = [
        mpatches.Patch(color='#F0F0F0', label='0 - Ô sạch / Máy'),
        mpatches.Patch(color='#F4D03F', label='1 - Bụi'),
    ]
    ax.legend(handles=legend, loc='upper right',
              bbox_to_anchor=(1.35, 1.02), fontsize=9)

    plt.tight_layout()
    plt.show()


# ── Khởi tạo ──
grid = create_env(ROWS, COLS, DUST_PROB)
total_dust = int(np.sum(grid == 1))

print(f"Ma trận ban đầu  |  Tổng bụi: {total_dust} ô")
draw_grid(grid, pos=(-1, -1), title=f'Ma trận ban đầu — Bụi: {total_dust} ô')


In [ ]:
def get_path_to_dust_ids_early(grid, start_pos):
    """Cách 1: Early Goal Test (Xét điều kiện rồi mới đưa vào frontier)"""
    rows, cols = grid.shape
    max_depth = rows * cols
    
    for limit in range(max_depth):
        if grid[start_pos[0]][start_pos[1]] == 1:
            return [start_pos]
            
        stack = [(start_pos, [start_pos])]
        while stack:
            (r, c), path = stack.pop()
            
            if len(path) - 1 < limit:
                # Đưa vào ngăn xếp theo thứ tự Phải, Trái, Xuống, Lên -> khi pop sẽ ưu tiên Lên, Xuống, Trái, Phải
                for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in path:
                        # 1. Xét điều kiện ngay trước khi đưa vào frontier
                        if grid[nr][nc] == 1:
                            return path + [(nr, nc)]
                        # 2. Không phải là đích, đưa vào frontier
                        stack.append(((nr, nc), path + [(nr, nc)]))
    return []

def get_path_to_dust_ids_late(grid, start_pos):
    """Cách 2: Late Goal Test (Đưa vào frontier, khi nào pop (lấy ra) mới xét)"""
    rows, cols = grid.shape
    max_depth = rows * cols
    
    for limit in range(max_depth):
        stack = [(start_pos, [start_pos])]
        while stack:
            (r, c), path = stack.pop()
            
            # 1. Lấy ra khỏi frontier rồi mới xét điều kiện
            if grid[r][c] == 1:
                return path
                
            if len(path) - 1 < limit:
                for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in path:
                        # 2. Trực tiếp đưa vào frontier mà không xét trước
                        stack.append(((nr, nc), path + [(nr, nc)]))
    return []


In [ ]:
def run_agent_ids(grid_in, mode='EARLY'):
    grid = grid_in.copy()
    rows, cols = grid.shape
    steps = 0
    cleaned = 0
    curr_pos = (0, 0)
    
    print('=' * 50)
    if mode == 'EARLY':
        print('CHẠY THUẬT TOÁN IDS (CÁCH 1: XÉT ĐIỀU KIỆN RỒI MỚI ĐƯA VÀO FRONTIER)')
    else:
        print('CHẠY THUẬT TOÁN IDS (CÁCH 2: ĐƯA VÀO FRONTIER RỒI MỚI XÉT)')
    print('=' * 50)
    print(f'Bắt đầu tại {curr_pos}')
    
    if grid[curr_pos[0]][curr_pos[1]] == 1:
        grid[curr_pos[0]][curr_pos[1]] = 0
        cleaned += 1
        print(f'   ⟹  Phát hiện bụi! Hút bụi tại {curr_pos} — Đã hút: {cleaned}/{total_dust}')
        
    while cleaned < total_dust:
        if mode == 'EARLY':
            path = get_path_to_dust_ids_early(grid, curr_pos)
        else:
            path = get_path_to_dust_ids_late(grid, curr_pos)
            
        if not path:
            print("Không thể tìm thấy thêm bụi nào!")
            break
            
        for next_pos in path[1:]:
            steps += 1
            r, c = next_pos
            pr, pc = curr_pos
            if r > pr: direction = 'XUỐNG'
            elif r < pr: direction = 'LÊN'
            elif c > pc: direction = 'PHẢI'
            else: direction = 'TRÁI'
            
            curr_pos = next_pos
            move_msg = f'Bước {steps}: Di chuyển {direction} → ô ({r}, {c})'
            
            if grid[r][c] == 1:
                grid[r][c] = 0
                cleaned += 1
                action_msg = f'   ⟹  Phát hiện bụi! Hút bụi tại ({r}, {c}) — Đã hút: {cleaned}/{total_dust}'
            else:
                action_msg = f'   ⟹  Ô sạch, tiếp tục.'
                
            print(move_msg)
            print(action_msg)
            
            # draw_grid(grid, pos=(r, c), title=f'Bước {steps} | Vị trí: ({r},{c}) | Đã hút: {cleaned}/{total_dust}')

    print('=' * 45)
    if cleaned == total_dust:
        status = 'THÀNH CÔNG'
        reason = 'Đã tìm và hút sạch hết bụi'
    else:
        status = 'THẤT BẠI'
        reason = 'Không hút hết bụi'

    print(f'Số bước đi  : {steps}')
    print(f'Bụi đã hút  : {cleaned} / {total_dust} ô')
    print(f'Trạng thái  : {status}')
    print(f'Lý do       : {reason}\n')


In [ ]:
# Chạy cả 2 cách trên cùng một môi trường
run_agent_ids(grid, mode='EARLY')
run_agent_ids(grid, mode='LATE')
